# arc3-v22-ab-tp10 — single-variable A/B: stock vs TP10 on the 27B V22 serving stack (one boot, two 25-game phases)

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/taaf-duck-qwen38-serving-v1", "driessmit1/arc3-vllm-h100-wheelhouse-v3"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")


# V22 only modifies the vLLM launch flags inside the bundled setup command.
# The target model owns native MTP tensors and the original checkpoint validation
# already verifies that mtp.* tensors are mounted.
def _v31_serving_commands(command: str) -> tuple[str, str, bool]:
    if "vllm.entrypoints.openai.api_server" not in command:
        return command, command, False

    source_block = """        '--kv-cache-dtype',
        'fp8',
    ]"""

    v22_block = """        '--kv-cache-dtype',
        'fp8',
        '--speculative-config',
        '{"method":"mtp","num_speculative_tokens":3}',
        '--async-scheduling',
    ]"""

    v31_block = """        '--kv-cache-dtype',
        'fp8',
        '--speculative-config',
        '{"method":"mtp","num_speculative_tokens":3}',
        '--async-scheduling',
        '--no-enable-chunked-prefill',
    ]"""

    if source_block not in command:
        print("V31: known source serving block not found; leaving it unchanged.", flush=True)
        return command, command, False

    return (
        command.replace(source_block, v31_block, 1),
        command.replace(source_block, v22_block, 1),
        True,
    )


def _v22_cleanup_partial_server() -> None:
    import signal

    pid_path = WORKING_DIR / "vllm-openai-server.pid"
    if not pid_path.exists():
        return
    try:
        pid = int(pid_path.read_text(encoding="utf-8").strip())
        try:
            os.kill(pid, signal.SIGTERM)
            time.sleep(2)
        except OSError:
            pass
        try:
            os.kill(pid, 0)
        except OSError:
            pass
        else:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
    except Exception as exc:
        print(f"V22: partial-server cleanup warning: {exc!r}", flush=True)
    finally:
        pid_path.unlink(missing_ok=True)


# Solver setup commands run before the benchmark loads.
# Primary = exact V22 MTP3 stack + no-chunked-prefill.
# Fallback = exact MTP3+async serving that produced 2.66 LB.
env = _command_env()
for original_command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    primary_command, v22_command, optimized = _v31_serving_commands(original_command)

    if not optimized:
        print(f"taaf.kaggle: setup command: {original_command}", flush=True)
        subprocess.run(original_command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    else:
        print("V31: launching MTP3 + async + FP8 KV + no-chunked-prefill.", flush=True)
        try:
            subprocess.run(primary_command, shell=True, check=True, cwd=WORKING_DIR, env=env)
        except subprocess.CalledProcessError as exc:
            print(
                f"V31: primary startup failed ({exc!r}); "
                "falling back to exact V22 MTP3+async.",
                flush=True,
            )
            _v22_cleanup_partial_server()
            subprocess.run(v22_command, shell=True, check=True, cwd=WORKING_DIR, env=env)

    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# ---- V31 runtime resilience ----
import threading as _v31_threading
import urllib.request as _v31_urllib

_V31_URL = os.environ.get("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1").rstrip("/")
_V31_PID = WORKING_DIR / "vllm-openai-server.pid"
_V31_LOG = WORKING_DIR / "vllm-openai-server.log"
_V31_STOP = _v31_threading.Event()
_V31_THREAD = None
_V31_LOCK = _v31_threading.Lock()


def _v31_healthy(timeout: float = 4.0) -> bool:
    try:
        with _v31_urllib.urlopen(f"{_V31_URL}/models", timeout=timeout) as response:
            return 200 <= int(response.status) < 500
    except Exception:
        return False


def _v31_wait_healthy(timeout: float = 480.0) -> bool:
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if _v31_healthy(5.0):
            return True
        time.sleep(5)
    return False


def _v31_kill() -> None:
    if not _V31_PID.exists():
        return
    try:
        pid = int(_V31_PID.read_text(encoding="utf-8").strip())
        try:
            os.kill(pid, signal.SIGTERM)
            time.sleep(2)
        except OSError:
            pass
        try:
            os.kill(pid, 0)
        except OSError:
            pass
        else:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
    except Exception as exc:
        print(f"V31 cleanup warning: {exc!r}", flush=True)
    finally:
        _V31_PID.unlink(missing_ok=True)


def _v31_spawn(no_chunked: bool) -> None:
    provenance_file = WORKING_DIR / "qwen38-model-provenance.json"
    provenance = json.loads(provenance_file.read_text(encoding="utf-8"))
    model_path = str(provenance["model_path"])

    site_packages = WORKING_DIR / "vllm-site-packages"
    child_env = os.environ.copy()
    current_pp = child_env.get("PYTHONPATH", "")
    if str(site_packages) not in current_pp.split(os.pathsep):
        child_env["PYTHONPATH"] = (
            str(site_packages)
            if not current_pp
            else str(site_packages) + os.pathsep + current_pp
        )
    child_env.update({
        "USE_TF": "0",
        "TRANSFORMERS_NO_TF": "1",
        "TRANSFORMERS_NO_TORCHVISION": "1",
        "VLLM_NO_USAGE_STATS": "1",
    })

    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_path,
        "--served-model-name", os.environ.get("INFERENCE_ANALYZER_MODEL", "Qwen/Qwen3.8-27B-FP8"),
        "--host", "127.0.0.1",
        "--port", "1234",
        "--tensor-parallel-size", "1",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "qwen3_coder",
        "--generation-config", "vllm",
        "--enable-prefix-caching",
        "--default-chat-template-kwargs", '{"preserve_thinking": true}',
        "--reasoning-parser", "qwen3",
        "--max-model-len", "262144",
        "--kv-cache-dtype", "fp8",
        "--speculative-config", '{"method":"mtp","num_speculative_tokens":3}',
        "--async-scheduling",
    ]
    if no_chunked:
        cmd.append("--no-enable-chunked-prefill")

    # Prevent unbounded log growth across restarts.
    if _V31_LOG.exists():
        previous = WORKING_DIR / "vllm-openai-server.previous.log"
        try:
            previous.unlink(missing_ok=True)
            _V31_LOG.replace(previous)
        except OSError:
            pass

    log_handle = _V31_LOG.open("w", encoding="utf-8")
    process = subprocess.Popen(
        cmd,
        env=child_env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    _V31_PID.write_text(str(process.pid), encoding="utf-8")

    if not _v31_wait_healthy():
        raise RuntimeError("restarted vLLM did not become healthy in time")


def _v31_recover() -> bool:
    with _V31_LOCK:
        if _v31_healthy():
            return True

        print("V31 watchdog: recovering MTP3/no-chunk server.", flush=True)
        _v31_kill()
        try:
            _v31_spawn(no_chunked=True)
            print("V31 watchdog: primary recovery succeeded.", flush=True)
            return True
        except Exception as first:
            print(f"V31 watchdog: primary recovery failed: {first!r}", flush=True)

        _v31_kill()
        try:
            _v31_spawn(no_chunked=False)
            print("V31 watchdog: exact V22 MTP3 recovery succeeded.", flush=True)
            return True
        except Exception as second:
            print(f"V31 watchdog: fallback recovery failed: {second!r}", flush=True)
            _v31_kill()
            return False


def _v31_start_watchdog(solver) -> None:
    global _V31_THREAD
    _V31_STOP.clear()

    def worker():
        failures = 0
        while not _V31_STOP.wait(15):
            if _v31_healthy():
                failures = 0
                continue

            failures += 1
            print(f"V31 watchdog: health failure {failures}/3", flush=True)
            if failures < 3:
                continue

            if _v31_recover():
                failures = 0
                continue

            print(
                "V31 watchdog: server unrecoverable; stopping solver to preserve partial score.",
                flush=True,
            )
            stop_event = getattr(solver, "_stop_event", None)
            if stop_event is not None:
                stop_event.set()
            return

    _V31_THREAD = _v31_threading.Thread(
        target=worker,
        name="v31-watchdog",
        daemon=True,
    )
    _V31_THREAD.start()


def _v31_stop_watchdog() -> None:
    _V31_STOP.set()
    if _V31_THREAD is not None:
        _V31_THREAD.join(timeout=5)


if not _v31_healthy(10):
    raise RuntimeError("V31 preflight: local vLLM API is not healthy.")
print("V31 preflight: local vLLM API healthy.", flush=True)



In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# V31 keeps the V22 gameplay/prompt/tool policy unchanged.
bm.solver.save_request_logs = False
print("V31 concurrency:", getattr(bm.solver, "concurrency", None))
print("V31 analyzer_timeout:", getattr(bm.solver, "analyzer_timeout", None))
print("V31 save_request_logs:", getattr(bm.solver, "save_request_logs", None))


In [ ]:
# ==== graft install (A/B machinery; every flag phase-controlled) ====
# All grafts install ONCE; behaviour is env-gated per phase. Defaults ALL OFF —
# the stock phase must be byte-equivalent stock behaviour.
import importlib as _il

for _flag in ['TP_ENABLE', 'TP2_ENABLE', 'TP4_ENABLE', 'TP5_ENABLE', 'TP6_ENABLE', 'TP7_ENABLE', 'TP8_ENABLE', 'TP9_ENABLE', 'TP10_ENABLE']:
    os.environ[_flag] = "0"

_G_DIR = WORKING_DIR / "graft_bundle"
_G_DIR.mkdir(parents=True, exist_ok=True)
_GRAFT_SOURCES = {'graft_memoryspine.py': '"""Memory-spine graft (TP10, 2026-08-31) — persistent notes, note-to-self echo,\nhonest last-turn accounting. Rank-2 in the path-to-7 plan.\n\nDesign source: docs/research-2026-08-31/R11-schema-traces-mining.md §3 (the\n95-99% Schema harness re-injects the agent\'s own notes.md every turn, echoes\nits prior intent/note-to-self verbatim, and reports honest last-turn\naccounting). Why it should pay here: R9 §2 forensics — cross-attempt amnesia\n(sp80: 15 attempts, fresh exploration each time) and the 08-29 review\'s 43%\nzero-level plays that are memory/control-bound, not action-starved.\n\nSTOCK BEHAVIOUR (verified in the bundled tool_agent.py) and the TRUE DELTA:\n\n(a) NOTES. Stock already harvests labelled world-model text from the\n    assistant channel (_extract_scientist_note :412, applied :1335) and\n    re-injects the LIVE values into every user prompt\n    (_summarized_knowledge_lines :1358, used :1472) — reinjection per turn is\n    NOT the gap. The gaps are: each harvest OVERWRITES the field (no history);\n    _update_summarized_knowledge_from_step_summary :1343 WIPES every field\n    except cross_level_notes on level_transition/run_complete/game_over\n    (graft_throughput\'s TP_KEEP_NOTES_ON_GAME_OVER suppresses only the\n    game_over wipe); and _ensure_session :1140 clears everything on a session\n    change, so nothing survives into a new pass of the same game.\n    DELTA: a module-level per-GAME journal, synced by diffing\n    _summarized_knowledge (so it also catches graft_emission\'s direct\n    reasoning-channel writes), snapshotted BEFORE the wipe and BEFORE a\n    session reset, and re-injected as a capped tail (TP10_NOTES_CAP, default\n    4000 chars) under "YOUR NOTES (persistent):". Entries identical to the\n    live block are skipped, so on healthy turns the block only carries what\n    stock has lost (earlier levels, pre-wipe state, earlier passes).\n\n(b) SUGGESTION ECHO. Stock maps "Plan:"/"Next test:" into current_plan —\n    also overwritten and wiped; there is no verbatim echo. DELTA: harvest the\n    last one-line `Next:` / `Suggestion:` from assistant text and echo it\n    verbatim at the TOP of the next user prompt ("YOUR PRIOR INTENT: ...");\n    consumed after one echo so a stale note is never re-served as fresh.\n    J10-F1: the harvested line is STRIPPED from the text handed to the stock\n    harvest — _extract_labeled_blocks (:375-:408) glues any unlabeled line\n    into the preceding labeled block, so an unstripped `Next:` line would be\n    absorbed into current_plan/world_model and re-served every turn as a\n    standing plan (and journaled as a stale imperative).\n\n(c) HONEST ACCOUNTING. Stock\'s prompt header (:1416) reports executed count\n    and names but never committed-vs-executed: requested_count /\n    stopped_early / state ARE recorded per action() payload (:329-:338,\n    :1674-:1682) and then dropped by _summarize_step_sequence (:1250) —\n    stop_reason and level ARE kept by the stock summary (:1288, :1293).\n    DELTA: augment the step summary with the committed total and end state\n    from the SAME recorded payloads, and prepend one line:\n    "LAST TURN: committed N action(s), M executed, ended level=L, state=S."\n    plus how many were dropped and the recorded stop_reason when a batch was\n    cut short. No mispredict detection is invented — only recorded fields.\n    J10-F2: requested_count is computed AFTER graft_throughput\'s batch-cap\n    truncation (:1760, :1890 sit downstream of _normalize_python_actions,\n    which TP wraps to truncate), so requested_count alone under-reports what\n    the model committed. TP10 therefore also wraps _normalize_python_actions\n    — whose stock either normalizes EVERY item or raises, never partially\n    drops — to record the RAW batch size per action() call (works in both\n    install orders because TP\'s cap wrapper hands the raw value to the inner\n    chain before truncating), and reports cap-truncated actions explicitly\n    ("N truncated by the harness batch cap").\n\nSeams (all rebinding; stock callables in the module-level _STOCK dict so\nstacked wrappers survive later installs):\n  ToolAgent._build_user_prompt                       — inject (a)(b)(c)\n  ToolAgent._update_summarized_knowledge_from_assistant — harvest (a)(b),\n                                                       intent line stripped first\n  ToolAgent._update_summarized_knowledge_from_step_summary — pre-wipe snapshot\n  ToolAgent._ensure_session                          — game key + pre-reset snapshot\n  ToolAgent._summarize_step_sequence                 — committed/state fields\n  ToolAgent._normalize_python_actions                — raw pre-clamp batch size\n  ToolAgent._run_python_tool                         — per-tool-call raw-count reset\n\nFlags (read at call time): TP10_ENABLE=1 master ("0" = pass-through);\nTP10_NOTES=1, TP10_ECHO=1, TP10_ACCOUNTING=1 per behaviour;\nTP10_NOTES_CAP=2000 chars of injected notes tail (J10-F4: R11 §3 prescribes\n<=2KB for the A/B; 4000 measured -4 retained history turns under 32k);\nTP10_HOWTO_EVERY=8 — the note-to-self how-to line rides prompt 1, every Nth\nprompt after, and the first prompt after a knowledge wipe (0 = wipe-only).\nFail-open: every graft path is try/except\'d back to stock output.\n"""\nfrom __future__ import annotations\n\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\n_STATE = {"installed": False}\n_STOCK: dict[str, Any] = {}\n_OFF = {"0", "false", "no", "off"}\n\n# game_key -> chronological journal of {"key","label","level","text"}\n_NOTES: dict[str, list[dict[str, Any]]] = {}\n_JOURNAL_MAX = 500\n_JOURNAL_TRIM_TO = 400\n_INTENT_MAX = 300\n\n_LABELS = (\n    ("world_model", "World model"),\n    ("goal_model", "Goal model"),\n    ("action_model", "Action model"),\n    ("recent_findings", "Recent findings"),\n    ("open_questions", "Open questions"),\n    ("current_plan", "Plan"),\n    ("cross_level_notes", "Cross-level notes"),\n)\n\nNOTES_HEADER = (\n    "YOUR NOTES (persistent): saved from your earlier world-model updates in THIS game; "\n    "they survive GAME_OVER and level changes. Reuse them instead of re-discovering."\n)\nECHO_LINE = (\n    "YOUR PRIOR INTENT (your note-to-self from last turn — reconsider, don\'t just obey): {intent}"\n)\nECHO_HOWTO = (\n    "To leave a note-to-self for your next turn, put one line starting `Next:` "\n    "(or `Suggestion:`) in your assistant text; it will be echoed back to you verbatim."\n)\n\n\n# ----------------------------------------------------------------- flags ---\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP10_ENABLE", "1").lower() not in _OFF\n\n\ndef notes_enabled() -> bool:\n    return enabled() and _env("TP10_NOTES", "1").lower() not in _OFF\n\n\ndef echo_enabled() -> bool:\n    return enabled() and _env("TP10_ECHO", "1").lower() not in _OFF\n\n\ndef accounting_enabled() -> bool:\n    return enabled() and _env("TP10_ACCOUNTING", "1").lower() not in _OFF\n\n\ndef notes_cap() -> int:\n    try:\n        return max(200, int(_env("TP10_NOTES_CAP", "2000")))\n    except ValueError:\n        return 2000\n\n\ndef howto_every() -> int:\n    try:\n        return max(0, int(_env("TP10_HOWTO_EVERY", "8")))\n    except ValueError:\n        return 8\n\n\ndef status() -> dict[str, Any]:\n    return {\n        "installed": _STATE["installed"],\n        "enabled": enabled(),\n        "notes": notes_enabled(),\n        "echo": echo_enabled(),\n        "accounting": accounting_enabled(),\n        "notes_cap": notes_cap(),\n        "howto_every": howto_every(),\n        "games_tracked": len(_NOTES),\n    }\n\n\n# ------------------------------------------------------------- game key ---\ndef _game_key(state_path: Any) -> str:\n    p = Path(state_path)\n    stem = p.stem  # e.g. "<artifact_stem>_p0_tool_runtime_state"\n    base = re.sub(r"_p\\d+_.*$", "", stem)\n    if base == stem:\n        base = re.sub(r"_p\\d+$", "", stem)\n    return f"{p.parent}::{base or stem}"\n\n\ndef game_key_of(agent: Any) -> str:\n    key = getattr(agent, "_tp10_game_key", None)\n    return key if key else f"agent-{id(agent)}"\n\n\ndef _summary_level(agent: Any) -> int | None:\n    try:\n        summary = getattr(agent, "_last_step_summary", None) or {}\n        level = summary.get("level")\n        return int(level) if level is not None else None\n    except Exception:  # noqa: BLE001\n        return None\n\n\n# --------------------------------------------------------- notes journal ---\ndef _latest_for_key(journal: list[dict[str, Any]], key: str) -> dict[str, Any] | None:\n    for entry in reversed(journal):\n        if entry.get("key") == key:\n            return entry\n    return None\n\n\ndef sync_journal(agent: Any, level: int | None) -> None:\n    """Diff the live _summarized_knowledge into the per-game journal.\n\n    Diff-based (not harvest-hook-based) so notes written by ANY channel —\n    stock assistant harvest, graft_emission\'s reasoning-channel fallback,\n    direct writes — are captured before a wipe can destroy them.\n    """\n    if not notes_enabled():\n        return\n    try:\n        knowledge = getattr(agent, "_summarized_knowledge", None)\n        if not isinstance(knowledge, dict):\n            return\n        journal = _NOTES.setdefault(game_key_of(agent), [])\n        for key, label in _LABELS:\n            text = str(knowledge.get(key) or "").strip()\n            if not text:\n                continue\n            latest = _latest_for_key(journal, key)\n            if latest is not None and latest.get("text") == text:\n                continue\n            journal.append({"key": key, "label": label, "level": level, "text": text})\n        if len(journal) > _JOURNAL_MAX:\n            del journal[: len(journal) - _JOURNAL_TRIM_TO]\n    except Exception:  # noqa: BLE001\n        pass\n\n\ndef render_notes(agent: Any) -> str | None:\n    journal = _NOTES.get(game_key_of(agent))\n    if not journal:\n        return None\n    live = getattr(agent, "_summarized_knowledge", None) or {}\n    seen: set[tuple[Any, Any]] = set()\n    keys_seen: set[Any] = set()\n    selected: list[dict[str, Any]] = []\n    for entry in reversed(journal):  # newest -> oldest\n        key = entry.get("key")\n        level = entry.get("level")\n        # J10-F3: a pre-first-summary (level=None) note is superseded by ANY\n        # newer entry for the same key — don\'t re-serve refuted early guesses\n        # beside their own corrections.\n        if level is None and key in keys_seen:\n            continue\n        ident = (key, level)\n        if ident in seen:\n            continue\n        seen.add(ident)\n        keys_seen.add(key)\n        # already shown verbatim in the live world-model block -> skip\n        if str(live.get(entry.get("key")) or "").strip() == entry.get("text"):\n            continue\n        selected.append(entry)\n    if not selected:\n        return None\n    selected.reverse()  # chronological, newest last\n    lines = []\n    for entry in selected:\n        level = entry.get("level")\n        tag = f"[L{level}] " if level is not None else ""\n        lines.append(f"- {tag}{entry.get(\'label\')}: {entry.get(\'text\')}")\n    cap = notes_cap()\n    kept: list[str] = []\n    total = 0\n    trimmed = False\n    for line in reversed(lines):  # keep the newest tail within the cap\n        if kept and total + len(line) + 1 > cap:\n            trimmed = True\n            break\n        if not kept and len(line) + 1 > cap:\n            line = line[: max(0, cap - 16)].rstrip() + "... [truncated]"\n        kept.append(line)\n        total += len(line) + 1\n    kept.reverse()\n    header = NOTES_HEADER + (" (older notes trimmed)" if trimmed else "")\n    return header + "\\n" + "\\n".join(kept)\n\n\n# ------------------------------------------------------- suggestion echo ---\ndef split_intent(content: str) -> tuple[str, str | None]:\n    """(text with `Next:`/`Suggestion:` lines removed, last one-line note).\n\n    J10-F1: the intent line MUST be removed from the text the stock harvest\n    sees — _extract_labeled_blocks glues unlabeled lines into the preceding\n    labeled block, so an unstripped note-to-self would be absorbed into\n    current_plan/world_model, re-served every turn, and journaled.\n    """\n    if not content or not content.strip():\n        return content, None\n    found: str | None = None\n    kept: list[str] = []\n    for raw_line in content.splitlines():\n        line = raw_line.strip()\n        while line[:1] in {"-", "*"}:\n            line = line[1:].lstrip()\n        lowered = line.lower()\n        matched = False\n        for prefix in ("next:", "suggestion:"):\n            if lowered.startswith(prefix):\n                matched = True\n                value = line[len(prefix):].strip()\n                if value:\n                    found = value\n                break\n        if not matched:\n            kept.append(raw_line)\n    if found and len(found) > _INTENT_MAX:\n        found = found[:_INTENT_MAX].rstrip() + "..."\n    return ("\\n".join(kept), found) if found is not None else (content, None)\n\n\ndef harvest_intent(content: str) -> str | None:\n    """Last one-line `Next:` / `Suggestion:` note in the assistant text."""\n    return split_intent(content)[1]\n\n\n# ----------------------------------------------------- honest accounting ---\ndef accounting_line(summary: Any) -> str | None:\n    """One line from the recorded per-turn payload fields only."""\n    if not isinstance(summary, dict) or not summary:\n        return None\n    try:\n        executed = int(summary.get("executed_count") or 0)\n    except (TypeError, ValueError):\n        return None\n    try:\n        committed = int(summary.get("tp10_committed"))\n    except (TypeError, ValueError):\n        committed = executed\n    committed = max(committed, executed)\n    level = summary.get("level")\n    state = summary.get("tp10_state")\n    if not state:\n        if summary.get("game_over"):\n            state = "GAME_OVER"\n        elif summary.get("run_complete"):\n            state = "WIN"\n        else:\n            state = "NOT_FINISHED"\n    line = (\n        f"LAST TURN: committed {committed} action(s), {executed} executed, "\n        f"ended level={level}, state={state}."\n    )\n    dropped = committed - executed\n    if dropped > 0:\n        try:\n            cap_dropped = min(dropped, max(0, int(summary.get("tp10_cap_dropped") or 0)))\n        except (TypeError, ValueError):\n            cap_dropped = 0\n        details = []\n        if cap_dropped > 0:\n            details.append(f"{cap_dropped} truncated by the harness batch cap")\n        reason = str(summary.get("stop_reason") or "").strip()\n        if reason and dropped - cap_dropped > 0:\n            details.append(f"stop_reason={reason}")\n        line += f" {dropped} committed action(s) were dropped before execution"\n        line += f" ({\'; \'.join(details)})." if details else "."\n    return line\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "memoryspine: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"memoryspine: SKIP (import failed: {exc!r})"\n    cls = getattr(agent_mod, "ToolAgent", None)\n    if cls is None:\n        return "memoryspine: SKIP (ToolAgent missing)"\n    for name in (\n        "_build_user_prompt",\n        "_update_summarized_knowledge_from_assistant",\n        "_update_summarized_knowledge_from_step_summary",\n        "_ensure_session",\n        "_summarize_step_sequence",\n        "_normalize_python_actions",\n        "_run_python_tool",\n    ):\n        if getattr(cls, name, None) is None:\n            return f"memoryspine: SKIP ({name} seam missing)"\n\n    _STOCK["build_user_prompt"] = cls._build_user_prompt\n    _STOCK["update_from_assistant"] = cls._update_summarized_knowledge_from_assistant\n    _STOCK["update_from_step_summary"] = cls._update_summarized_knowledge_from_step_summary\n    _STOCK["ensure_session"] = cls._ensure_session\n    _STOCK["summarize_step_sequence"] = cls._summarize_step_sequence\n    _STOCK["normalize_python_actions"] = cls._normalize_python_actions\n    _STOCK["run_python_tool"] = cls._run_python_tool\n\n    # -- seam: session identity + pre-reset snapshot ------------------------\n    def ensure_session(self, state_path):\n        try:\n            if enabled():\n                old_dir = getattr(self, "_session_runtime_dir", None)\n                new_dir = getattr(Path(state_path), "parent", None)\n                if old_dir is not None and new_dir is not None and old_dir != new_dir:\n                    # stock is about to clear _summarized_knowledge: snapshot\n                    # it under the OLD game key first.\n                    sync_journal(self, _summary_level(self))\n        except Exception:  # noqa: BLE001\n            pass\n        result = _STOCK["ensure_session"](self, state_path)\n        try:\n            self._tp10_game_key = _game_key(state_path)\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    ensure_session._tp10_stock = _STOCK["ensure_session"]\n    cls._ensure_session = ensure_session\n\n    # -- seam: harvest (notes sync + intent) --------------------------------\n    def update_from_assistant(self, content):\n        text = content\n        intent = None\n        if enabled() and echo_enabled():\n            try:\n                # J10-F1: remove the note-to-self line BEFORE the stock\n                # harvest sees it, or _extract_labeled_blocks glues it into\n                # the preceding labeled block for the rest of the game.\n                stripped, intent = split_intent(str(content or ""))\n                if intent is not None:\n                    text = stripped\n            except Exception:  # noqa: BLE001\n                text, intent = content, None\n        result = _STOCK["update_from_assistant"](self, text)\n        if not enabled():\n            return result\n        try:\n            if intent:\n                self._tp10_intent = intent\n            sync_journal(self, _summary_level(self))\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    update_from_assistant._tp10_stock = _STOCK["update_from_assistant"]\n    cls._update_summarized_knowledge_from_assistant = update_from_assistant\n\n    # -- seam: pre-wipe snapshot --------------------------------------------\n    def update_from_step_summary(self):\n        if enabled():\n            try:\n                summary = getattr(self, "_last_step_summary", None) or {}\n                level = _summary_level(self)\n                if summary.get("level_transition") and isinstance(level, int) and level > 1:\n                    # the knowledge being wiped described the level just left\n                    level = level - 1\n                sync_journal(self, level)\n                if summary.get("level_transition") or summary.get("run_complete") or summary.get("game_over"):\n                    # J10-F4: re-teach the note-to-self affordance on the\n                    # first prompt after a knowledge wipe\n                    self._tp10_wipe_pending = True\n            except Exception:  # noqa: BLE001\n                pass\n        return _STOCK["update_from_step_summary"](self)\n\n    update_from_step_summary._tp10_stock = _STOCK["update_from_step_summary"]\n    cls._update_summarized_knowledge_from_step_summary = update_from_step_summary\n\n    # -- seam: raw pre-clamp batch size (J10-F2) ----------------------------\n    # graft_throughput\'s batch cap truncates inside its _normalize_python_actions\n    # wrapper, and requested_count (:1760, :1890) is computed AFTER that — so\n    # requested_count alone under-reports what the model committed. Stock\n    # normalize either accepts every item or raises (:1614-:1650, no partial\n    # drop), and TP\'s cap wrapper hands the raw value to the inner chain\n    # before truncating, so this wrapper sees the true batch size in BOTH\n    # install orders. Recorded only when the chain returns (a cap REFUSAL\n    # raises out of the outer wrapper and the model gets the explicit error).\n    def normalize_python_actions(self, value):\n        result = _STOCK["normalize_python_actions"](self, value)\n        if enabled():\n            try:\n                if isinstance(value, (list, tuple)):\n                    raw = len(value)\n                elif isinstance(value, (str, dict)):\n                    raw = 1\n                else:\n                    raw = len(result)\n                calls = getattr(self, "_tp10_raw_committed", None)\n                if not isinstance(calls, list):\n                    # lazily create so the capture also works when a caller\n                    # bypasses _run_python_tool (the run wrapper still resets\n                    # the list at the top of every real tool call)\n                    calls = []\n                    self._tp10_raw_committed = calls\n                calls.append(max(int(raw), len(result)))\n            except Exception:  # noqa: BLE001\n                pass\n        return result\n\n    normalize_python_actions._tp10_stock = _STOCK["normalize_python_actions"]\n    cls._normalize_python_actions = normalize_python_actions\n\n    def run_python_tool(self, state_path, arguments):\n        try:\n            self._tp10_raw_committed = []\n        except Exception:  # noqa: BLE001\n            pass\n        return _STOCK["run_python_tool"](self, state_path, arguments)\n\n    run_python_tool._tp10_stock = _STOCK["run_python_tool"]\n    cls._run_python_tool = run_python_tool\n\n    # -- seam: committed/state ride the recorded step summary ---------------\n    def summarize_step_sequence(self, action_results):\n        summary = _STOCK["summarize_step_sequence"](self, action_results)\n        if not enabled() or summary is None:\n            return summary\n        try:\n            committed = 0\n            state = None\n            for item in action_results or []:\n                if not isinstance(item, dict):\n                    continue\n                requested = item.get("requested_count")\n                if requested is None:\n                    requested = item.get("executed_count")\n                if requested is None:\n                    requested = 1\n                try:\n                    committed += max(0, int(requested))\n                except (TypeError, ValueError):\n                    committed += 1\n                if item.get("executed") and item.get("state") is not None:\n                    state = item.get("state")\n            # J10-F2: prefer the true pre-clamp count captured at the\n            # normalize seam; the difference is what the batch cap discarded.\n            raw_calls = getattr(self, "_tp10_raw_committed", None)\n            if isinstance(raw_calls, list) and raw_calls:\n                raw_total = sum(int(n) for n in raw_calls)\n                if raw_total > committed:\n                    summary["tp10_cap_dropped"] = raw_total - committed\n                    committed = raw_total\n            summary["tp10_committed"] = committed\n            if state is not None:\n                summary["tp10_state"] = str(state)\n        except Exception:  # noqa: BLE001\n            pass\n        return summary\n\n    summarize_step_sequence._tp10_stock = _STOCK["summarize_step_sequence"]\n    cls._summarize_step_sequence = summarize_step_sequence\n\n    # -- seam: prompt injection ---------------------------------------------\n    def build_user_prompt(self, action_num, *args, **kwargs):\n        text = _STOCK["build_user_prompt"](self, action_num, *args, **kwargs)\n        if not enabled():\n            return text\n        try:\n            prefix: list[str] = []\n            if accounting_enabled():\n                line = accounting_line(getattr(self, "_last_step_summary", None))\n                if line:\n                    prefix.append(line)\n            if echo_enabled():\n                intent = getattr(self, "_tp10_intent", None)\n                if intent:\n                    prefix.append(ECHO_LINE.format(intent=intent))\n                    self._tp10_intent = None  # one echo per note; never re-serve stale intent\n            suffix: list[str] = []\n            if notes_enabled():\n                # final sync catches direct writes (e.g. TP5 reasoning harvest)\n                sync_journal(self, _summary_level(self))\n                block = render_notes(self)\n                if block:\n                    suffix.append(block)\n            if echo_enabled():\n                # J10-F4: the how-to line is ~150 chars on EVERY turn if\n                # unconditional; ride prompt 1, every Nth prompt, and the\n                # first prompt after a knowledge wipe.\n                count = int(getattr(self, "_tp10_prompt_count", 0) or 0) + 1\n                self._tp10_prompt_count = count\n                every = howto_every()\n                periodic = every > 0 and (count - 1) % every == 0\n                if periodic or bool(getattr(self, "_tp10_wipe_pending", False)):\n                    suffix.append(ECHO_HOWTO)\n                    self._tp10_wipe_pending = False\n            if not prefix and not suffix:\n                return text\n            return "\\n".join([*prefix, text, *suffix])\n        except Exception:  # noqa: BLE001\n            return text\n\n    build_user_prompt._tp10_stock = _STOCK["build_user_prompt"]\n    cls._build_user_prompt = build_user_prompt\n\n    _STATE["installed"] = True\n    return "memoryspine: OK"\n'}
for _name, _src in _GRAFT_SOURCES.items():
    (_G_DIR / _name).write_text(_src, encoding="utf-8")
if str(_G_DIR) not in sys.path:
    sys.path.insert(0, str(_G_DIR))
_installed = {}
for _mod_name in ['graft_memoryspine']:
    _m = _il.import_module(_mod_name)
    _st = _m.install()
    _installed[_mod_name] = _st
    assert _st.endswith(": OK"), _st
print("[ab] grafts installed:", _installed, flush=True)


In [ ]:
# ==== two-phase A/B run: stock then tp10 (one boot, same server) ====
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))
assert not TRUE_SUBMISSION, "A/B smoke kernel must never run as a submission"

_ENV_DIR = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
PHASES = [
    ("stock", {f: "0" for f in ['TP_ENABLE', 'TP2_ENABLE', 'TP4_ENABLE', 'TP5_ENABLE', 'TP6_ENABLE', 'TP7_ENABLE', 'TP8_ENABLE', 'TP9_ENABLE', 'TP10_ENABLE']}),
    ("tp10", dict({f: "0" for f in ['TP_ENABLE', 'TP2_ENABLE', 'TP4_ENABLE', 'TP5_ENABLE', 'TP6_ENABLE', 'TP7_ENABLE', 'TP8_ENABLE', 'TP9_ENABLE', 'TP10_ENABLE']}, **{'TP10_ENABLE': '1'})),
]
AB_RESULTS = {}
_PHASE_BUDGET_S = 4.0 * 3600

_v31_start_watchdog(bm.solver)
try:
    for _phase_name, _phase_env in PHASES:
        for _k, _v in _phase_env.items():
            os.environ[_k] = _v
        bm.game_runs = []
        bm.games = _offline_games(_ENV_DIR)
        bm.n_passes = 1
        bm.game_weights = None
        bm.label = "v22-ab-" + _phase_name
        _soft_end = datetime.now() + timedelta(seconds=_PHASE_BUDGET_S)
        print(f"=== PHASE {_phase_name}: {len(bm.games)} games env={_phase_env} "
              f"soft_end={_soft_end} ===", flush=True)
        try:
            await bm.run(soft_end_time=_soft_end, runtime_environment=target,
                         minimal_diagnostics=False)
        except Exception as _exc:  # noqa: BLE001
            import traceback

            print(f"PHASE {_phase_name} RAISED {type(_exc).__name__}: {_exc}", flush=True)
            traceback.print_exc()
        games = []
        for game_run in list(bm.game_runs):
            apl = list(game_run.actions_per_level or [])
            games.append({
                "game_id": game_run.game_id,
                "state": str(game_run.state),
                "levels_completed": game_run.levels_completed,
                "number_of_levels": game_run.number_of_levels,
                "final_score": game_run.final_score,
                "actions": sum(apl) if apl else len(game_run.history),
                "actions_per_level": apl,
                "wallclock_s": game_run.final_wallclock_seconds,
            })
        n = max(1, len(games))
        AB_RESULTS[_phase_name] = {
            "games": games,
            "n": len(games),
            "mean_score": round(sum(g["final_score"] for g in games) / n, 3),
            "mean_levels": round(sum(g["levels_completed"] for g in games) / n, 3),
            "zero_level": sum(1 for g in games if not g["levels_completed"]),
            "total_actions": sum(g["actions"] for g in games),
        }
        (WORKING_DIR / "ab_results.json").write_text(json.dumps(AB_RESULTS, indent=1))
        print(f"=== PHASE READ {_phase_name}: mean_score={AB_RESULTS[_phase_name]['mean_score']} "
              f"mean_levels={AB_RESULTS[_phase_name]['mean_levels']} "
              f"zero={AB_RESULTS[_phase_name]['zero_level']}/{n} "
              f"actions={AB_RESULTS[_phase_name]['total_actions']} ===", flush=True)
finally:
    _v31_stop_watchdog()
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

print("\n==== A/B SUMMARY (tp10 vs stock) ====")
for _p, _r in AB_RESULTS.items():
    print(f"{_p:8s} mean_score={_r['mean_score']} mean_levels={_r['mean_levels']} "
          f"zero={_r['zero_level']}/{_r['n']} actions={_r['total_actions']}")
